# BlinkIQ: Business Intelligence Dashboard for Retail Sales Analytics

## Notebook 1: Data Understanding

### Objective
The objective of this notebook is to understand the Blinkit Grocery Sales dataset before data cleaning and analysis. This includes examining the dataset structure, identifying missing values, checking for duplicates, understanding data types, and gathering initial observations.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_excel("../data/Tableau BlinkIT Grocery Project U16955293080 (4).xlsx")

In [3]:
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [4]:
# Shape of the dataset
print("Shape:", df.shape)

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Dataset information
print("\nDataset Info:")
df.info()

Shape: (8523, 12)

Columns:
['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'Item_Outlet_Sales']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outle

In [5]:
# Check missing values
missing_values = df.isnull().sum().sort_values(ascending=False)

print(missing_values)

Outlet_Size                  2410
Item_Weight                  1463
Item_Fat_Content                0
Item_Identifier                 0
Item_Visibility                 0
Item_Type                       0
Outlet_Identifier               0
Item_MRP                        0
Outlet_Establishment_Year       0
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64


In [6]:
# Missing values in percentage
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_percentage = missing_percentage.sort_values(ascending=False)

print(missing_percentage)

Outlet_Size                  28.276428
Item_Weight                  17.165317
Item_Fat_Content              0.000000
Item_Identifier               0.000000
Item_Visibility               0.000000
Item_Type                     0.000000
Outlet_Identifier             0.000000
Item_MRP                      0.000000
Outlet_Establishment_Year     0.000000
Outlet_Location_Type          0.000000
Outlet_Type                   0.000000
Item_Outlet_Sales             0.000000
dtype: float64


In [7]:
# Check duplicate records
duplicates = df.duplicated().sum()

print("Duplicate Records:", duplicates)

Duplicate Records: 0


In [8]:
# Check unique values in categorical columns

categorical_columns = [
    'Item_Fat_Content',
    'Item_Type',
    'Outlet_Size',
    'Outlet_Location_Type',
    'Outlet_Type'
]

for col in categorical_columns:
    print("=" * 50)
    print(col)
    print(df[col].unique())

Item_Fat_Content
['Low Fat' 'Regular' 'low fat' 'LF' 'reg']
Item_Type
['Dairy' 'Soft Drinks' 'Meat' 'Fruits and Vegetables' 'Household'
 'Baking Goods' 'Snack Foods' 'Frozen Foods' 'Breakfast'
 'Health and Hygiene' 'Hard Drinks' 'Canned' 'Breads' 'Starchy Foods'
 'Others' 'Seafood']
Outlet_Size
['Medium' nan 'High' 'Small']
Outlet_Location_Type
['Tier 1' 'Tier 3' 'Tier 2']
Outlet_Type
['Supermarket Type1' 'Supermarket Type2' 'Grocery Store'
 'Supermarket Type3']


In [9]:
# Standardize Item_Fat_Content values

df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})

# Verify the changes
print(df['Item_Fat_Content'].unique())

['Low Fat' 'Regular']


In [10]:
# Fill missing Item_Weight using the average weight of the same Item_Identifier

df['Item_Weight'] = df['Item_Weight'].fillna(
    df.groupby('Item_Identifier')['Item_Weight'].transform('mean')
)

# Check if any missing values remain
print("Missing Item_Weight:", df['Item_Weight'].isnull().sum())

Missing Item_Weight: 4


In [11]:
# Fill remaining missing values with the overall median

df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].median())

# Verify
print("Missing Item_Weight:", df['Item_Weight'].isnull().sum())

Missing Item_Weight: 0


In [12]:
# Fill Outlet_Size using the mode of each Outlet_Type

df['Outlet_Size'] = df.groupby('Outlet_Type')['Outlet_Size'].transform(
    lambda x: x.fillna(x.mode()[0])
)

# Check remaining missing values
print("Missing Outlet_Size:", df['Outlet_Size'].isnull().sum())

Missing Outlet_Size: 0


In [13]:
# Check zero values in Item_Visibility

zero_visibility = (df['Item_Visibility'] == 0).sum()

print("Zero Visibility Values:", zero_visibility)

Zero Visibility Values: 526


In [14]:
# Replace 0 values with NaN
df['Item_Visibility'] = df['Item_Visibility'].replace(0, np.nan)

# Fill missing values using the average visibility of the same product
df['Item_Visibility'] = df['Item_Visibility'].fillna(
    df.groupby('Item_Identifier')['Item_Visibility'].transform('mean')
)

# Fill any remaining missing values with the overall median
df['Item_Visibility'] = df['Item_Visibility'].fillna(df['Item_Visibility'].median())

# Verify
print("Zero Visibility Values:", (df['Item_Visibility'] == 0).sum())
print("Missing Visibility Values:", df['Item_Visibility'].isnull().sum())

Zero Visibility Values: 0
Missing Visibility Values: 0


In [15]:
# Final check for missing values

print(df.isnull().sum())

Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64


In [16]:
# Save cleaned dataset
df.to_csv("../data/blinkit_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [17]:
# ===============================
# KPI Summary
# ===============================

total_sales = df['Item_Outlet_Sales'].sum()
average_sales = df['Item_Outlet_Sales'].mean()
total_products = df['Item_Identifier'].nunique()
total_outlets = df['Outlet_Identifier'].nunique()
average_mrp = df['Item_MRP'].mean()

print("Total Sales:", round(total_sales,2))
print("Average Sales:", round(average_sales,2))
print("Unique Products:", total_products)
print("Total Outlets:", total_outlets)
print("Average MRP:", round(average_mrp,2))

Total Sales: 18591125.41
Average Sales: 2181.29
Unique Products: 1559
Total Outlets: 10
Average MRP: 140.99


In [18]:
# Number of unique values

print(df.nunique())

Item_Identifier              1559
Item_Weight                   446
Item_Fat_Content                2
Item_Visibility              8322
Item_Type                      16
Item_MRP                     5938
Outlet_Identifier              10
Outlet_Establishment_Year       9
Outlet_Size                     3
Outlet_Location_Type            3
Outlet_Type                     4
Item_Outlet_Sales            3493
dtype: int64
